# 品质理论：转移矩阵、纯回收机与组装-回收环

本文档用非具体化的代数参数给出 Factorio 品质系统的数学模型，分三章：

1. 基础转移矩阵（品质升级规则与生产矩阵）；
2. 纯回收机循环（单设备自循环，传奇产品收集）；
3. 组装机 + 回收机循环（10 维状态，块矩阵闭式解）。

约定：全部使用**行向量右乘**的约定，即状态为行向量 $x$（$1\times n$），一步演化写作 $x' = x\,M$，矩阵元 $M_{ij}$ 表示“从等级 $i$ 转移到等级 $j$ 的概率 $\times$ 出入比例”。若使用列向量约定，等价模型为 $x' = M^{\mathsf T} x$，所有公式中的矩阵整体转置即可；2.2 与 docs/theory/00-notation.md 会标注原文中由此产生的转置差异。

所有推导只给结论式（闭式 + 级数），不展开运算细节。具体数值验证放在第 3 步的模拟代码中完成（数值锚点与勘误见 docs/theory/00-notation.md）。

## 1 基础转移矩阵

### 1.1 两个表达式

输出 = 输入 × 转移矩阵：

$$g = f\,A$$

转移矩阵 = 出入比例 × 品质矩阵：

$$A = \mathrm{diag}(r)\,Q(q)$$

记号与参数来源见 1.1.1；状态向量的含义见 1.1.2；常见特例见 1.1.3；$Q(q)$ 的显式矩阵见 1.2，滚动机制与分段通式见 1.2.1。

### 1.1.1 记号与约定

- **品质等级**：叙述一律用 $1$–$5$ 的等级编号——$1$=普通、$2$=优秀、$3$=稀有、$4$=史诗、$5$=传奇（口语“$k$ 星”即等级 $k$）；代码实现里再用 $0$-index（$0..4$）与之对应。
- $i \in \{1,2,3,4,5\}$：品质等级指标，记传奇档 $L := 5$。
- $q$：单次制作的品质概率（最终 roll 概率），由设备插槽上的品质插件加成叠加而来：若有 $4$ 个插槽、每个品质插件提供 $2.5\%$，则 $q = 4\times 2.5\% = 10\%$，即 $q=\text{插槽数}\times\text{单插件倍率}$（同规格插件时；不同规格取各自倍率之和）。
- $r$：**出入比例**——产出组数 / 投入组数（已含全部产能来源，按“一组”计，与配方件数无关，例如 $1$ 铜块 $\to$ $2$ 铜线的配方 $r$ 依然是 $1$）。$r$ 的来源枚举如下：
  1. **设备产能**：部分设备自带产能，如铸造厂、电磁工厂；
  2. **产能插件**：插在设备上的产能插件加成；
  以上两项合并计入组装侧：$r = 1+P$，其中 $P$ 为“设备自带 + 插件”的总产能加成（如 $50\%$ 产能即 $P=0.5$）；
  3. **回收机**：固定保留 $\tfrac14$（湮灭 $\tfrac34$），即 $r = \tfrac14$（游戏定值，不参数化）。
- $e_i$：第 $i$ 个分量为 $1$ 的列向量（$5\times 1$）；$e_L$ 即传奇收集指示向量；$\mathbf 1$ 为全 $1$ 列向量。
- 流体不受品质机制约束——**不是**令 $q=0$，而是在本理论中**没有任何表现**：没有任何表达式与流体相关。即使有一天要考虑流体，这里也不会变化——$1$ 份原材料 $	o$ $1$ 份产品仍然是 $A = \mathrm{diag}(1)$。此处仅作游戏机制的提示。

### 1.1.2 状态向量与转移矩阵

$f = (f_1,f_2,f_3,f_4,f_5)$：$f_k$ 为等级 $k$ 的物品量。单位是“**组**”——产出 $1$ 件产品所需的若干原料算 $1$ 组（如 $1$ 铜块 $\to$ $2$ 铜线时，$2$ 根铜线为 $1$ 组），因此量与配方件数无关。

$A$ 的第 $i$ 行第 $j$ 列：输入品质 $i$ 时产出品质 $j$ 的**期望倍数**（$=$ 概率 $\times$ 出入比例）。$g = fA$ 即作用一次后各等级的量；多级设备串联即矩阵连乘，循环系统展开为无限长直线链（2.3、3.4）。

### 1.1.3 特例：普通生产、组装机与回收机的基础矩阵

在 $g = fA$ 中取 $A = \mathrm{diag}(r)\,Q(q)$，按 $r$ 的来源（1.1.1）特例化：

- **普通生产**（无品质、无产能）：$A = I = \mathrm{diag}(1)$，品质与数量都不变。
- **组装机**：$A = \mathrm{diag}(1+P)\,Q(q)$，$P$ 为设备自带产能（铸造厂、电磁工厂等）与产能插件的合计。
- **回收机**：$A = \mathrm{diag}(\tfrac14)\,Q(q)$，固定湮灭 $\tfrac34$（游戏定值，不参数化）。

三者都可直接左乘状态：$g = fA$。可作为练习取具体 $f$ 与 $q$ 手算一遍 $g$，并检验 $Q(q)$ 行和为 $1$（品质滚动本身不改变件数，变化只来自 $r$）。

若各等级配置不同（如低等级挂品质插件、高等级挂产能插件），可逐行取 $(q_i, r_i)$；细节留待模拟代码建模。

### 1.2 品质矩阵 $Q(q)$

$$
Q(q) =
\begin{pmatrix}
1-q & 9q\cdot10^{-1} & 9q\cdot10^{-2} & 9q\cdot10^{-3} & q\cdot10^{-3}\\
0 & 1-q & 9q\cdot10^{-1} & 9q\cdot10^{-2} & q\cdot10^{-2}\\
0 & 0 & 1-q & 9q\cdot10^{-1} & q\cdot10^{-1}\\
0 & 0 & 0 & 1-q & q\\
0 & 0 & 0 & 0 & 1
\end{pmatrix}.
$$

滚动机制、分段通式与校验见 1.2.1。

### 1.2.1 滚动机制、分段通式与校验

**滚动机制**（官方）：先以概率 $q$ 首次判定；成功则升一档，再以固定 $10\%$ 的概率继续判定，直到失败；传奇为吸收态；**不可能跳档**（连续升级是逐档累乘 $1/10$ 的几何因子，而非品质加成的幂次，见 docs/theory/00-notation.md 勘误 2）。

由此得到分段通式：

$$
Q(q)_{ij} =
\begin{cases}
0, & j < i,\\[4pt]
1-q, & j = i,\\[4pt]
\dfrac{9q}{10}\cdot 10^{-(j-i-1)}, & i < j < L,\\[6pt]
q\cdot 10^{-(L-i-1)}, & j = L > i.
\end{cases}
$$

以等级 $1$（普通）行为例，与 wiki 表格逐项一致：升 1 档 $\tfrac{9}{10}q$，升 2 档 $\tfrac{9}{100}q$，升 3 档 $\tfrac{9}{1000}q$，直升传奇 $\tfrac{1}{1000}q$。

**注意**：$Q(q)$ 是一般转移矩阵 $A$ 的特例——只有品质滚动、出入比例 $r\equiv 1$；加上产能或湮灭后即 $A = \mathrm{diag}(r)\,Q(q)$，常见特例见 1.1.3。

### 1.3 范围：不考虑速度与节能插件

模型只考虑**品质插件**与**产能插件**（分别进入 $q$ 与 $r$，见 1.1.1）。

- **速度插件**：不考虑。它只会降低 $q$（挤占品质插件的槽位），对 $r$ 没有影响；吞吐不足完全可以用**多铺设备**补足，速度只是工程侧变量。
- **节能插件**：不考虑。电能消耗不需要优化，只要**供电充足**即可，模型不引入能耗约束。

## 2 纯回收机循环

模型背景：把物品反复送入回收机，目标是收集传奇产物（例如“铁板 $\to$ 传奇铁板”的自循环）。回收机湮灭 $3/4$ 的物品，未湮灭的以品质分布返回，传奇则被移出系统。

### 2.1 设定与湮灭率

- 回收机固定保留 $\tfrac14$（湮灭 $\tfrac34$）：游戏定值、无需参数化；组归一化下与配方件数无关。
- 回收机的品质概率 $q$ 由品质插件决定（见 1.1.1）。
- 进料为行向量 $f$（$1\times 5$，通常 $f_L = 0$；若 $f_L>0$，传奇进料会因下行清零而直接离开系统，即被收集）。

### 2.2 收集矩阵与回收矩阵

系统把**收集**（传奇产品移出系统）与**继续回收**（非传奇部分再过机）分开处理。定义**收集矩阵**（Collect）

$$
C := \mathrm{diag}(0,0,0,0,1),
$$

它只保留传奇（5 星）分量。$g = fA$ 是产出向量，则 $gC = fAC$ 就是产出中的传奇产品向量；剩余部分用**通过掩码**表示

$$
S := I - C = \mathrm{diag}(1,1,1,1,0).
$$

**回收矩阵**（$R$ 取 **recycle** 的首字母；不预先清行）为

$$
R := \tfrac14\, Q(q).
$$

一次过机的分解：产出为 $fR$，其中被收集走的是 $fRC$，继续留在循环里的是 $fRS$。

（行向量约定下掩码体现为右乘以筛选分量；列向量约定下对应形式为左乘，见 docs/theory/00-notation.md 勘误 4。）


### 2.3 循环展开、闭式与数值截断

对画图与模拟，循环用编译原理的**循环展开**处理：把回收机的逐次通过展开成有限长的链即可，

$$
fR \;\to\; \underbrace{fRC}_{\text{收集}} \;+\; \underbrace{fRS}_{\text{继续}},
\qquad
fRSR \;\to\; \underbrace{fRSRC}_{\text{收集}} \;+\; \underbrace{fRSRS}_{\text{继续}},\quad \cdots
$$

代数式则直接求和。总收集（各等级的传奇产出向量 $\mathbf y$）为

$$
\mathbf y \;=\; fRC + fRSRC + fRSRSRC + \cdots \;=\; fR\bigl(I + SR + (SR)^2 + \cdots\bigr)C
\;=\; fR\,(I - SR)^{-1}\,C,
$$

标量产出 $y = \mathbf y\,\mathbf 1 = fR(I-SR)^{-1}C\,\mathbf 1$，非传奇存量为 $s_k = f(SR)^k$（$(SR)^k$ 的传奇分量恒为 $0$）。

因为每过一次机都有 $\tfrac34$ 被湮灭，$(SR)^k$ 的行和以 $(\tfrac14)^k$ 几何衰减：$\|SR\|_\infty \le \|R\|_\infty \le \tfrac14 < 1$，级数收敛，链“不那么长”。数值模拟可以很快截断（逐段累加直到余项小于容差）；矩阵等比数列求和 $(I-SR)^{-1}$ 与截断等价，**实际实现建议用数值截断**。

## 3 组装机 + 回收机循环

模型背景：组装机把材料制成产品，非传奇产品送回收机拆回材料，材料再回组装机；只把**传奇产品**移出系统（传奇材料继续作为组装机输入，最终仍以传奇产品形式离开）。

### 3.1 设定：10 维状态

- 组装机矩阵 $A$（$5\times 5$，材料档位 $\to$ 产品档位）：$A = \mathrm{diag}(r_A)\cdot Q(q_A)$，$r_A = 1+P$（组归一化；跨计量基准时为 $\frac{o_A}{i_A}(1+P)$），$q_A$ 为组装机的最终品质概率。
- 回收机矩阵 $R$（$5\times 5$，产品档位 $\to$ 材料档位）：$R = \tfrac14\, Q(q)$（同 2.2，不预先清行）；收集矩阵 $C = \mathrm{diag}(0,0,0,0,1)$，传奇**产品**被收集、不再回流，非传奇产品先乘 $S = I - C$ 再进回收机，传奇**材料**列保留（继续进组装机）。
- 状态为 $1\times 10$ 行向量 $x = [\,m \mid p\,]$，$m$ 为材料五档计数，$p$ 为产品五档计数。

### 3.2 轮转矩阵与奇偶分解

$$
T :=
\begin{pmatrix}
0 & A\\
SR & 0
\end{pmatrix}
\in \mathbb{R}^{10\times 10},
\qquad
x' = x\,T .
$$

（行向量约定下**无需转置**；wiki 的 $[0,A;R,0]$ 形式在列向量约定下出现整体转置，见 docs/theory/00-notation.md 勘误 4。）由分块乘法：

$$
T^2 =
\begin{pmatrix}
ASR & 0\\
0 & SRA
\end{pmatrix},
\qquad
T^{2k} = \mathrm{diag}\bigl((ASR)^k,\ (SRA)^k\bigr),\quad
T^{2k+1} = T^{2k}\,T .
$$

取初值 $x_0 = [\,f \mid 0\,]$（只有材料进料），奇偶步自动分离：

$$
x_{2k} = \bigl[\,f(ASR)^k \mid 0\,\bigr],\qquad
x_{2k+1} = \bigl[\,0 \mid f(ASR)^k A\,\bigr].
$$

### 3.3 单环算子 $K = ASR$

把“组装 $\to$ 掩码 $\to$ 回收 $\to$ 掩码”压成一个 $5\times 5$ 算子：

$$
K := A\,S\,R, \qquad R = \tfrac14\, Q(q),\quad S = I - C .
$$

即：组装产出 $mA$，传奇产品被 $C$ 收集走，非传奇部分经 $S$ 回到回收机，回收产出直接成为下一轮组装材料（传奇材料保留）。且 $K$ 的行和 $\le (1+P)\cdot\tfrac14$。$K$ 就是“一轮组装-回收后材料的再分配”，材料演化为 $m_{k+1} = m_k K$。


### 3.4 闭式与级数

材料总吞吐、产品总产出、传奇产品产出：

$$
m = f\,(I-K)^{-1} = f\bigl(I + K + K^2 + \cdots\bigr),\qquad
p = m\,A = f\,(I-K)^{-1} A,
$$

$$
\boxed{\;y \;=\; f\,(I - K)^{-1} A\, e_L \;=\; f\,\bigl(I + K + K^2 + \cdots\bigr)\,A\,e_L\;}
$$

等价的 $10\times 10$ 块闭式（对 $I-T$ 求块逆，与 3.2 的奇偶分解一致）：

$$
(I-T)^{-1} =
\begin{pmatrix}
(I-ASR)^{-1} & (I-ASR)^{-1}A\\
(I-SRA)^{-1}SR & (I-SRA)^{-1}
\end{pmatrix},
\qquad
x_0\,(I-T)^{-1} = \bigl[\,f(I-ASR)^{-1} \mid f(I-ASR)^{-1}A\,\bigr].
$$

### 3.5 变体：同时收集传奇材料

若同时把传奇材料也移出系统（wiki 示例的做法），则回收产出也乘 $S$，收集用 $C = \mathrm{diag}(0,0,0,0,1)$：

$$
K' = A\,S\,R\,S,\qquad
y' = f\,(I-K')^{-1}\bigl(A\,C + A\,S\,R\,C\bigr)\mathbf 1 .
$$

原笔记中的总收集式 $f(AC + AMRA) + f(AMRM)(AC+AMRA)+\cdots$ 里，第二项应为**材料收集** $AMRC$ 而非 $AMRA$（见 docs/theory/00-notation.md 勘误 3）；修正后即上式的级数形式 $f\bigl(I + K' + K'^2 + \cdots\bigr)(AC + ASRC)\mathbf 1$。

### 3.6 收敛性

- 精确判据：$\rho(K) < 1$（谱半径）。
- 充分判据（行和界）：$\|K\|_\infty \le \|A\|_\infty\|R\|_\infty$（组装机行和 $1+P$，回收机行和 $\tfrac14$）。在组归一化下（组装与回收的物料单位一致）：

$$
(1+P)\cdot\tfrac14 \;=\; \frac{1+P}{4} \;<\; 1
\quad\Longleftrightarrow\quad
P < 300\%,
$$

即**总产能倍率 $1+P$ 必须小于 $4$（$400\%$）**，否则级数发散、系统无稳态。这与 wiki “总产能趋近 $400\%$ 时收敛变慢” 的临界一致：$P \to 300\%$ 时 $\|K\| \to 1$，级数收敛速度按 $\theta^k$ 变慢，其中 $\theta = (1+P)/4$。

- 逐行参数下（各档位 $q_i, r_i$ 不同）行和界不再充分，需直接判 $\rho(K)$；实践中可用 $\max_i \sum_j K_{ij}$ 作为快速上界估计。

### 3.7 守恒律与产出占比

记 $p_S := p\,S$（进入回收机的非传奇产品），则有三条可用于自检的不变量：

$$
m = f + p_S R \quad(\text{材料守恒：进料 + 回收产出 = 材料总吞吐}),
$$
$$
p = m A \quad(\text{产品由材料经组装机产生，含传奇}),
$$
$$
f\cdot\mathbf 1 \;=\; \tfrac34\, p_S\cdot\mathbf 1 \;+\; y \quad(\text{投入 = 湮灭 + 收集的传奇}).
$$

产出占比的代数定义（与进料归一化无关）：

$$
\eta \;=\; \frac{y}{f\cdot\mathbf 1}
\quad(\text{每单位原料组得到的传奇产品数}),\qquad
\mu \;=\; \frac{m\cdot\mathbf 1}{f\cdot\mathbf 1}
\quad(\text{内部放大倍数}).
$$

$\mu = \bigl\|(I-K)^{-1}\bigr\|$ 的量级在 $\theta \to 1$ 时发散为 $1/(1-\theta)$，这正是“产能越接近 $400\%$，系统吞吐爆炸越剧烈”的代数根源。